## Imports

In [1]:
import sys
import time
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.db import get_engine, run_query, fetch_table

print('Project root:', project_root)

Project root: d:\OMEN-16\ML-Projects\airline-satisfaction-ml


## Connecting to PostgreSQL

In [2]:
engine = get_engine()
print(run_query("select version() AS server_version",engine)["server_version"][0][:100])


PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


## Fetching the Whole Table

In [3]:
start = time.perf_counter()
df = fetch_table(engine)
seconds = time.perf_counter() - start

print("shape:", df.shape)
print("Loaded in",round(seconds,1), "seconds")

shape: (129880, 24)
Loaded in 0.9 seconds


In [4]:
df.head()

,id,gender,age,customer_type,type_of_travel,travel_class,flight_distance,departure_delay,arrival_delay,departure_arrival_time_convenience,...,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling,satisfaction
0,1,Male,48,First-time,Business,Business,821,2,5.0,3,...,3,5,2,5,5,5,3,5,5,Neutral or Dissatisfied
1,2,Female,35,Returning,Business,Business,821,26,39.0,2,...,5,4,5,5,3,5,2,5,5,Satisfied
2,3,Male,41,Returning,Business,Business,853,0,0.0,4,...,3,5,3,5,5,3,4,3,3,Satisfied
3,4,Male,50,Returning,Business,Business,1905,0,0.0,2,...,5,5,5,4,4,5,2,5,5,Satisfied
4,5,Female,49,Returning,Business,Business,3470,0,1.0,3,...,3,4,4,5,4,3,3,3,3,Satisfied


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 129880 entries, 0 to 129879
Data columns (total 24 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  129880 non-null  int64  
 1   gender                              129880 non-null  str    
 2   age                                 129880 non-null  int64  
 3   customer_type                       129880 non-null  str    
 4   type_of_travel                      129880 non-null  str    
 5   travel_class                        129880 non-null  str    
 6   flight_distance                     129880 non-null  int64  
 7   departure_delay                     129880 non-null  int64  
 8   arrival_delay                       129487 non-null  float64
 9   departure_arrival_time_convenience  129880 non-null  int64  
 10  ease_of_online_booking              129880 non-null  int64  
 11  checkin_service                     1

In [6]:
df.dtypes

id                                      int64
gender                                    str
age                                     int64
customer_type                             str
type_of_travel                            str
travel_class                              str
flight_distance                         int64
departure_delay                         int64
arrival_delay                         float64
departure_arrival_time_convenience      int64
ease_of_online_booking                  int64
checkin_service                         int64
online_boarding                         int64
gate_location                           int64
onboard_service                         int64
seat_comfort                            int64
leg_room_service                        int64
cleanliness                             int64
food_and_drink                          int64
inflight_service                        int64
inflight_wifi_service                   int64
inflight_entertainment            

## Missing Values

In [7]:
print("Missing values per column")
print(df.isnull().sum())

Missing values per column
id                                      0
gender                                  0
age                                     0
customer_type                           0
type_of_travel                          0
travel_class                            0
flight_distance                         0
departure_delay                         0
arrival_delay                         393
departure_arrival_time_convenience      0
ease_of_online_booking                  0
checkin_service                         0
online_boarding                         0
gate_location                           0
onboard_service                         0
seat_comfort                            0
leg_room_service                        0
cleanliness                             0
food_and_drink                          0
inflight_service                        0
inflight_wifi_service                   0
inflight_entertainment                  0
baggage_handling                        0
satisfac

## Validating the Fetch

In [8]:
pg_count = run_query("select count(*) as n from airline_passenger_satisfaction",engine)["n"][0]
print("Rows in postgreSQL :", pg_count)
print("Rows in DataFrame:", len(df))
print("match:", pg_count == len(df))

Rows in postgreSQL : 129880
Rows in DataFrame: 129880
match: True


## Loading the Original CSV for comparision

In [9]:
csv_df = pd.read_csv("../data/raw/airline_passenger_satisfaction.csv")

column_map = {
     "ID": "id", "Gender": "gender", "Age": "age", "Customer Type": "customer_type",
    "Type of Travel": "type_of_travel", "Class": "travel_class",
    "Flight Distance": "flight_distance", "Departure Delay": "departure_delay",
    "Arrival Delay": "arrival_delay",
    "Departure and Arrival Time Convenience": "departure_arrival_time_convenience",
    "Ease of Online Booking": "ease_of_online_booking",
    "Check-in Service": "checkin_service", "Online Boarding": "online_boarding",
    "Gate Location": "gate_location", "On-board Service": "onboard_service",
    "Seat Comfort": "seat_comfort", "Leg Room Service": "leg_room_service",
    "Cleanliness": "cleanliness", "Food and Drink": "food_and_drink",
    "In-flight Service": "inflight_service", "In-flight Wifi Service": "inflight_wifi_service",
    "In-flight Entertainment": "inflight_entertainment", "Baggage Handling": "baggage_handling",
    "Satisfaction": "satisfaction",
}

csv_df = csv_df.rename(columns = column_map)

print("same shape:", csv_df.shape == df.shape)
print("same column names:", list(csv_df.columns) == list(df.columns))

same shape: True
same column names: True


## CHecking the values

In [10]:
a = df.sort_values("id").reset_index(drop=True)
b = csv_df.sort_values("id").reset_index(drop=True)

try:
    pd.testing.assert_frame_equal(a,b,check_dtype=False)
    print("PostgreSQL data and CSV data are identical, value for value.")
except AssertionError as error:
    print("Difference found:")
    print(error)

PostgreSQL data and CSV data are identical, value for value.


## Querying from python

In [11]:
query = """
select travel_class,
       count(*) as passengers,
       round(100.0 * AVG(case when
                                 satisfaction = 'Satisfied' then 1 else 0 end),1)
                                 as pct_satisfied
from airline_passenger_satisfaction
group by travel_class
order by pct_satisfied desc
"""
run_query(query,engine)


,travel_class,passengers,pct_satisfied
0,Business,62160,69.4
1,Economy Plus,9411,24.6
2,Economy,58309,18.8


In [12]:
run_query(
    """
    select id,age,travel_class,flight_distance,satisfaction
    from airline_passenger_satisfaction
    where travel_class = :cls and age >= :min_age
    order by flight_distance desc
    limit 5
    """,
    engine,
    params = {"cls": "Business","min_age": 60},
)

,id,age,travel_class,flight_distance,satisfaction
0,31870,63,Business,4983,Satisfied
1,56476,60,Business,4963,Satisfied
2,58700,60,Business,4243,Satisfied
3,96325,60,Business,3998,Satisfied
4,82695,72,Business,3996,Neutral or Dissatisfied


## comparision of pandas and sql results

In [13]:
same_in_pandas = df[df["travel_class"] == "Business"].shape[0]
same_in_sql = run_query(
    "select count(*) as n from airline_passenger_satisfaction where travel_class = 'Business'", engine)["n"][0]
print("Business-class rows,counted in pandas:", same_in_pandas)
print("Business-class rows, counted in sql:", same_in_sql)
    

Business-class rows,counted in pandas: 62160
Business-class rows, counted in sql: 62160


## Creating the backup copy

In [14]:
df_raw  = df.copy()
print("Backup made:", df_raw.shape)

Backup made: (129880, 24)
